In [1]:
pip install kaggle 


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [kaggle]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install kaggle


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
mkdir -p experiments/zyra_training/data/raw

In [2]:
!kaggle datasets download -d ronakbokaria/myntra-products-dataset -p ../data/raw

Dataset URL: https://www.kaggle.com/datasets/ronakbokaria/myntra-products-dataset
License(s): CC0-1.0
100%|███████████████████████████████████████| 115M/115M [00:26<00:00, 4.59MB/s]



In [3]:
import zipfile 


Extracted: myntra-products-dataset.zip


In [4]:
for path in raw.rglob("*"):
    if path.is_file():
        print(path)

../data/raw/.DS_Store
../data/raw/myntra-products-dataset.zip
../data/raw/myntra-products-dataset/myntra202305041052.csv


In [12]:
import pandas as pd 
csv = "../data/raw/myntra-products-dataset/myntra202305041052.csv"
df = pd.read_csv(csv)
print("Shape:", df.shape)
df.head(10)
missing = pd.DataFrame({
    "missing": df.isnull().sum(),
    "missing_%": (df.isnull().mean() * 100).round(2),
    "dtype": df.dtypes
})

display(missing)
df.info()
missing = pd.DataFrame({
    "missing": df.isnull().sum(),
    "missing_%": (df.isnull().mean() * 100).round(2),
    "dtype": df.dtypes
})

display(missing)
print(df.columns.tolist())

Shape: (1060213, 11)


,missing,missing_%,dtype
id,0,0.0,int64
name,0,0.0,str
img,0,0.0,str
asin,0,0.0,str
price,0,0.0,float64
mrp,0,0.0,float64
rating,0,0.0,float64
ratingTotal,0,0.0,int64
discount,0,0.0,int64
seller,0,0.0,str


<class 'pandas.DataFrame'>
RangeIndex: 1060213 entries, 0 to 1060212
Data columns (total 11 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   id           1060213 non-null  int64  
 1   name         1060213 non-null  str    
 2   img          1060213 non-null  str    
 3   asin         1060213 non-null  str    
 4   price        1060213 non-null  float64
 5   mrp          1060213 non-null  float64
 6   rating       1060213 non-null  float64
 7   ratingTotal  1060213 non-null  int64  
 8   discount     1060213 non-null  int64  
 9   seller       1060213 non-null  str    
 10  purl         1060213 non-null  str    
dtypes: float64(3), int64(3), str(5)
memory usage: 89.0 MB


,missing,missing_%,dtype
id,0,0.0,int64
name,0,0.0,str
img,0,0.0,str
asin,0,0.0,str
price,0,0.0,float64
mrp,0,0.0,float64
rating,0,0.0,float64
ratingTotal,0,0.0,int64
discount,0,0.0,int64
seller,0,0.0,str


['id', 'name', 'img', 'asin', 'price', 'mrp', 'rating', 'ratingTotal', 'discount', 'seller', 'purl']


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1060213 entries, 0 to 1060212
Data columns (total 11 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   id           1060213 non-null  int64  
 1   name         1060213 non-null  str    
 2   img          1060213 non-null  str    
 3   asin         1060213 non-null  str    
 4   price        1060213 non-null  float64
 5   mrp          1060213 non-null  float64
 6   rating       1060213 non-null  float64
 7   ratingTotal  1060213 non-null  int64  
 8   discount     1060213 non-null  int64  
 9   seller       1060213 non-null  str    
 10  purl         1060213 non-null  str    
dtypes: float64(3), int64(3), str(5)
memory usage: 89.0 MB


In [13]:
print(df.columns.tolist())

['id', 'name', 'img', 'asin', 'price', 'mrp', 'rating', 'ratingTotal', 'discount', 'seller', 'purl']


In [15]:
# Basic dataset statistics

print("Rows:", len(df))
print("Unique IDs:", df["id"].nunique())
print("Unique ASINs:", df["asin"].nunique())
print("Unique Product Names:", df["name"].nunique())
print("Unique Sellers:", df["seller"].nunique())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["id"].duplicated().sum())
print("Duplicate ASINs:", df["asin"].duplicated().sum())
df.describe().T

Rows: 1060213
Unique IDs: 1060213
Unique ASINs: 1
Unique Product Names: 217738
Unique Sellers: 5496

Duplicate rows: 0
Duplicate IDs: 0
Duplicate ASINs: 1060212


,count,mean,std,min,25%,50%,75%,max
id,1060213.0,530107.000000,306057.274812,1.0,265054.0,530107.0,795160.0,1060213.0
price,1060213.0,1536.235155,3051.140381,25.0,499.0,811.0,1497.0,257500.0
mrp,1060213.0,2668.379334,3877.900441,25.0,999.0,1780.0,2999.0,257500.0
rating,1060213.0,1.060150,1.829632,0.0,0.0,0.0,2.8,5.0
ratingTotal,1060213.0,41.896183,747.664535,0.0,0.0,0.0,3.0,76400.0
discount,1060213.0,149.641952,564.878137,0.0,15.0,50.0,68.0,19996.0


In [17]:
images_per_product = df.groupby("id")["img"].nunique()

print("Average images per product:", images_per_product.mean())
print("Minimum:", images_per_product.min())
print("Maximum:", images_per_product.max())

display(images_per_product.value_counts().sort_index())

Average images per product: 1.0
Minimum: 1
Maximum: 1


img
1    1060213
Name: count, dtype: int64

In [21]:
from pathlib import Path

list(Path("../data/raw/myntra-fashion-products").rglob("*.csv"))



[]

In [22]:
!kaggle datasets download -d nirokey/myntra-fashion-products -p ../data/raw

Dataset URL: https://www.kaggle.com/datasets/nirokey/myntra-fashion-products
License(s): CC0-1.0
100%|█████████████████████████████████████| 2.96M/2.96M [00:01<00:00, 1.85MB/s]



In [47]:
from pathlib import Path

list(Path("../data/raw").glob("*"))
import zipfile
from pathlib import Path

zip_path = Path("../data/raw/myntra-fashion-products.zip")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("../data/raw/myntra-fashion-products")

print("Done")

Done


In [51]:
## P1 — Text quality

In [52]:
df2["name"].str.len().describe()
df2["description"].str.len().describe()

count    12491.000000
mean       159.206789
std        155.099275
min          7.000000
25%         91.000000
50%        119.000000
75%        171.000000
max       3670.000000
Name: description, dtype: float64

In [54]:
print(df2[df2["name"].str.len() < 10][["name", "description", "gender"]].to_string())
print(df2[df2["description"].str.len() < 20][["name", "description", "gender"]].to_string())

          name                                                         description gender
8378  Sherlock  Grey and Grey printed T-shirt, has a round neck, and short sleeves  Women
9529  Superman          Black printed T-shirt, has a round neck, and short sleeves    Men
                                                                   name          description gender
128    Difference of Opinion Men Olive Green Printed Round Neck T-shirt              T-shirt    Men
412                        Raymond Men Yellow Solid Polo Collar T-shirt              T-shirt    Men
837                        Raymond Men Orange Solid Polo Collar T-shirt              T-shirt    Men
1478                           Wild stone Men Stone Body Perfume 120 ml   Stone Body Perfume    Men
1794                           Wild Stone Men Edge Eau de Parfum 100 ml   Edge Eau de Parfum    Men
3637                       Smiley World Women Burgundy Solid Sweatshirt         Closed Front  Women
4043                          

# **P2 — Image count quality**

## We already know:

## Average ≈ 4.91
## Min = 1
## Max = 10
## 511 tested images → 0 failures
## All tested images = 1080 × 1440

## That's actually excellent.

# **P3 — Price quality**

In [60]:
print(df2["price"].describe())
print("Zero/negative prices:", (df2["price"] <= 0).sum())

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
count    12491.000000
mean      1452.660956
std       2118.503976
min         90.000000
25%        649.000000
50%        920.000000
75%       1499.000000
max      63090.000000
Name: price, dtype: float64
Zero/negative prices: 0


## **P4 — Stock**

In [61]:
print(df2["in_stock"].value_counts(dropna=False))

in_stock
True     12188
False      303
Name: count, dtype: int64


In [62]:
print(df2["in_stock"].value_counts(dropna=False))

in_stock
True     12188
False      303
Name: count, dtype: int64


In [63]:
print(ProductTextEncoderInput)
print(ProductAttributeEncoderInput)
print(ProductAttributes)
print(ProductImageEncoderInput)
print(ProductImageInput)

NameError: name 'ProductTextEncoderInput' is not defined